In [ ]:
pip install -r requirements.txt

In [2]:
import os
from time import sleep
import astral
import pytz
from astral.sun import sun
utc=pytz.UTC
import sys
import cmath
import datetime as dt
from math import cos,radians
from statistics import mean, stdev
import locale
import pandas as pd
import numpy as np
import geopandas as gp
import matplotlib.pyplot as plt
import matplotlib as mpl
import rasterio as rio
from rasterio.plot import show
from rasterio.mask import mask
from shapely.geometry import box, Point,LineString,MultiLineString,Polygon
import json as jsn
from fiona.errors import DriverError
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection, PatchCollection
from scipy.interpolate import UnivariateSpline, SmoothBivariateSpline

import glob
import seaborn as sns
import matplotlib.patches as patches
from matplotlib.path import Path
from matplotlib.patches import PathPatch
from matplotlib.collections import PatchCollection
from scipy import stats
from IPython.display import display, clear_output
import ipywidgets as ipw
workDir=os.getcwd()
print(workDir)

def createInputNams(gdf_cerf):
    animalTest=gdf_cerf.prenom.unique()[0]
    tmp=pd.DataFrame({"x":gdf_cerf.geometry.x, "y":gdf_cerf.geometry.y, "time":range(0,gdf_cerf.shape[0])})

    resultPath=os.path.join(workDir, animalTest)
    try:
        os.mkdir(resultPath)
    except OSError as error:
        print(error)   
    tmp.to_csv(os.path.join(resultPath,f"{animalTest}_input_file_for_VFractal_nams.csv"), index=False,header=False)

def mapSeasons(monthValue):
    corr={1:"Decembre-Fevrier",
          2:"Decembre-Fevrier",
          3:"Mars-Mai",
          4:"Mars-Mai",
          5:"Mars-Mai",
          6:"Juin-Aout",
          7:"Juin-Aout",
          8:"Juin-Aout",
          9:"Septembre-Novembre",
          10:"Septembre-Novembre",
          11:"Septembre-Novembre",
          12:"Decembre-Fevrier"}
    return corr[monthValue]   

def mapSeasonIndex(seasonValue):
  corr={"Juin-Aout":2,
  "Septembre-Novembre":3,
  "Decembre-Fevrier":4,
  "Mars-Mai":1}
  return corr[seasonValue]

def mapSexZH(animalID):
    corr={"Amanda":"femelle",
          "Beni":"male",
          "Christoph":"male",
          "Dario":"male",
          "Frieda":"femelle",
          "Erika":"femelle"
          }
        
    return corr[animalID]

def mapSexASB(animalID):
    corr={
"Joba":"male",
"Kaja": "male",
"Wika":"male",
"Ardy":"male",
"Bajo": "male",
"Sipe": "male",
"Yano":"male",
"Hota": "femelle",
"Kata":"femelle",
"Lola":"femelle",
"Luna": "femelle",
"Regina":"femelle",
"Rowa":"femelle",
"Zaja":"femelle",
"Scotch": "male",
"Aiko":"male",
"Arvi":"male",
"Bubu":"male",
"Eika":"femelle",
"Eiya":"femelle",
"Flick":"male",
"Foro":"male",
"Sala":"femelle",
"Hope":"femelle",
"Rody":"male",
"Ruma":"femelle",
"Sala":"femelle",
"Xina":"femelle",
"Zima":"femelle",
"Flin":"male"
  }
    return corr[animalID]

def mapSexVersoix(animalID):
  corr={"Baron":"male",
  "Bibiche":"femelle",
  "Biolina":"femelle",
  "Cocotte":"femelle",
  "Collioud":"male",
  "Courtois":"male",
  "Daille":"femelle",
  "Fabiola":"femelle",
  "Fabiolle":"femelle",
  "Julie":"femelle",
  "Junior":"male",
  "Miolan":"male",
  "Neige":"femelle",
  "Obelix":"male",
  "Prince":"male",
  "Rouphie":"femelle",
  "Tictac":"male",
  "Titan":"male"      }
  return corr[animalID]
def mapSexHauteSavoie(animalID):
    corr={
45558:"male",
45559: "femelle",
45560:"femelle",
45561:"femelle",
45562:"femelle",
45564: "femelle",
45565: "male",
45567:"femelle",
45568: "femelle",
45569:"femelle",
45570:"femelle",
45571: "daguet",
45809:"male",
45810:"male",
45811: "male"
  }
    return(corr[animalID])
def plot_polygon(ax, poly, **kwargs):
    path = Path.make_compound_path(
        Path(np.asarray(poly.exterior.coords)[:, :2]),
        *[Path(np.asarray(ring.coords)[:, :2]) for ring in poly.interiors])

    patch = PathPatch(path, **kwargs)
    collection = PatchCollection([patch], **kwargs)
    
    ax.add_collection(collection, autolim=True)
    ax.autoscale_view()
    return collection


def getDeerYear(gdf_cerf,yearList:list):
  deerYearVector=pd.Series(["NA" for i in range(len(gdf_cerf))])
  yearList.append(yearList[-1]+1)
  for i in yearList:
    if pd.to_datetime(i, format="%Y").is_leap_year:
        deerYear=[pd.Timestamp(i-1,3,1),pd.Timestamp(i,2,29)+pd.Timedelta(days=1)]
    else:
        deerYear=[pd.Timestamp(i-1,3,1),pd.Timestamp(i,2,28)+pd.Timedelta(days=1)]

    #deerYear=[pd.Timestamp(i-1,6,1),pd.Timestamp(i,5,31)+pd.Timedelta(days=1)]    
    
    for index, row in gdf_cerf.iterrows():
         if deerYear[0]<row.dateTime<=deerYear[1]:
              deerYearVector.iloc[index]=f"deerYear_{i-1}-{i}"
  return deerYearVector

def getSunPeriod(gdf_cerf:gp.GeoDataFrame, country:str):
    periodList=[]
    longLat=gdf_cerf.geometry.to_crs("epsg:4326")
    for index, row in gdf_cerf.iterrows():

        if country=="Switzerland":
            local_tz = pytz.timezone('Europe/Zurich')
            loc_cerf=astral.LocationInfo("Suisse",region="Switzerland",timezone="Europe/Zurich",latitude=longLat.iloc[index].y, longitude=longLat.iloc[index].x)
        elif country=="France":
            local_tz = pytz.timezone('Europe/Paris')
            loc_cerf=astral.LocationInfo("Haute-Savoie",region="France",timezone="Europe/Paris",latitude=longLat.iloc[index].y, longitude=longLat.iloc[index].x)
        else:
            print("no country was specified")
            raise Exception


        s = sun(loc_cerf.observer, date=row.dateTime)
        if local_tz.localize(row.dateTime)<=s["dawn"]:
            period="nuit"
        elif local_tz.localize(row.dateTime)<=s["sunrise"]:
            period="nuit"
        elif local_tz.localize(row.dateTime)<=s["sunset"]:
            period="jour"
        elif local_tz.localize(row.dateTime)<=s["dusk"]:
            period="jour"
        else:
            period="nuit"
        periodList.append(period)
    return periodList


/home/luvil/Workflow_fractalDeer/GPS_DATA_PARSERS


***
## LOADING RAW GPS DATA AND FORMATTING
***

### red deer from Bern

In [ ]:
animalTest = "Yano"

fileDataName="GPS_relocation_H.csv"
tmp=pd.read_csv(fileDataName)
tmp["dateTime"]=pd.to_datetime(tmp.datum+" "+tmp.zeit,format="%Y/%m/%d %H:%M:%S")
tmp["year"]=pd.Series(tmp.dateTime).dt.year
tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
tmp.rename({"Ind":"prenom"},axis=1, inplace=True)
tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
tmp["sexe"]=tmp["prenom"].apply(mapSexASB)


subset_cerf_1=tmp.loc[(tmp["prenom"]==animalTest)]
nbYear=subset_cerf_1["year"].unique().tolist()

subset_cerf=gp.GeoDataFrame({"prenom":subset_cerf_1.prenom,
                             "sexe":subset_cerf_1.sexe,
                             "annee":subset_cerf_1.year,
                             "mois":subset_cerf_1.mois,
                            "saison":subset_cerf_1.saison, 
                            "dateTime":subset_cerf_1.dateTime,
                            "geometry":gp.points_from_xy(subset_cerf_1["xcoord"], subset_cerf_1["ycoord"]),
                            "UTC_DATE":subset_cerf_1.dateTime.dt.date,
                            "UTC_TIME":subset_cerf_1.dateTime.dt.time,
                            "x_coord":subset_cerf_1.xcoord,
                            "y_coord":subset_cerf_1.ycoord},
                            crs="EPSG:2056").reset_index(drop=True)
del(tmp)
print(f"********** {animalTest} **********")
print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
print(subset_cerf["deerYear"].value_counts())
resultPath=os.path.join(workDir, animalTest)
try:
    os.mkdir(resultPath)
except OSError as error:
    print(error)
csvPath=os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv")
if not os.path.isfile(csvPath):
    subset_cerf.to_csv(os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv"), index=False)   
else:
    print("file already exists")

********** Yano **********
start : 2013-02-20 00:00:14, end : 2014-09-15 19:00:00 
deerYear_2013-2014    4939
deerYear_2012-2013     339
deerYear_2014-2015       6
Name: deerYear, dtype: int64


### red deer from FRIBOURG : a unique SHP file with every animals

In [10]:
workDir = "/home/luvil/IE-OFEV/Cerf_Fribourg"
overwrite = True
allDat=gp.read_file(os.path.join(workDir, "TELEMETRIE_CERF.shp"))

c=1
for animalName in set(allDat["INDIVIDU"]):
    if animalName == 'Flin':
        continue
    print(f"********** {animalName} **********")
    tmp=allDat.loc[allDat["INDIVIDU"]==animalName].copy()
   # tmp.dropna(axis=0, subset="acquisition_time", inplace=True)
   # tmp.dropna(axis=0, subset=["x_lv95","y_lv95"], inplace=True)
    tmp["dateTime"]=pd.to_datetime(tmp.DATE_+" "+tmp.TIME_, format="%Y-%m-%d %H:%M:%S")
    tmp["year"]=pd.Series(tmp.dateTime).dt.year
    tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
    tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
    tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
    tmp["sexe"]=tmp["INDIVIDU"].apply(mapSexASB)
    nbYear=tmp["year"].unique().tolist()


    subset_cerf=gp.GeoDataFrame({"prenom":tmp.INDIVIDU,
                                "sexe":tmp.sexe,
                                "annee":tmp.year,
                                "mois":tmp.mois,
                                "saison":tmp.saison, 
                                "heure":tmp.heure,
                                "dateTime":tmp.dateTime,
                                 "geometry":gp.points_from_xy(tmp["X_COORD"], tmp["Y_COORD"]),
                                "UTC_DATE":tmp.dateTime.dt.date,
                                "UTC_TIME":tmp.dateTime.dt.time
                                },
                                crs="EPSG:2056").reset_index(drop=True).sort_values(by="dateTime", axis=0)
    del(tmp)
    print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
    subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
    subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
    #print(subset_cerf["deerYear"].value_counts())


    gb = subset_cerf.groupby(['prenom', 'deerYear'])
    agregat=gb.agg(
    nbFixes=('dateTime', "count"),
    startDT=('dateTime', "min"),
    endDT=('dateTime', "max")
    ).reset_index()
    
    if c==1:
        finalStat=agregat.copy()
    else:
        finalStat=pd.concat([finalStat, agregat], ignore_index=True)
    c=c+1


    resultPath=os.path.join(workDir, str(animalName))
    try:
        os.mkdir(resultPath)
    except OSError as error:
        print(error)

    for yearTest in subset_cerf["deerYear"].unique():
        subYear=subset_cerf.loc[subset_cerf["deerYear"]==yearTest].reset_index(drop=True)
        subYear.sort_values(by="dateTime", axis=0,inplace=True)
        csvPath=os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv")
        if not os.path.isfile(csvPath) or overwrite:
            subYear.to_csv(os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv"), index=False)   
        else:
            print("file already exists : no overwriting")
    
    csvPath=os.path.join(resultPath,f"{animalName}_allFixes_formatted.csv")
    if not os.path.isfile(csvPath) or overwrite:
        subset_cerf.to_csv(os.path.join(resultPath,f"{animalName}_allFixes_formatted.csv"), index=False)   
        subset_cerf.drop(["UTC_DATE", "UTC_TIME"],axis=1).to_file(os.path.join(workDir,"RED_DEER_FRIB_allFixes_formatted.gpkg"), driver='GPKG', layer=str(animalName))
        
    else:
        print("file already exists : no overwriting")

finalStat["Duration_days"]=(finalStat["endDT"]-finalStat["startDT"]).dt.days
finalStat["is_full_year"]=finalStat["Duration_days"].apply(lambda x : "yes" if x>=364 else "no")
finalStat.to_csv(os.path.join(workDir,"RED_DEER_FRIB_fixes_counts.csv"), index=False)

********** Hota **********
start : 2011-03-07 06:48:09, end : 2012-09-07 19:03:05 
[Errno 17] File exists: '/home/luvil/IE-OFEV/Cerf_Fribourg/Hota'


/home/luvil/.local/lib/python3.8/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


********** Scotch **********
start : 2009-08-19 08:02:03, end : 2010-03-10 04:00:54 


/home/luvil/.local/lib/python3.8/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


********** Kata **********
start : 2009-02-24 19:03:24, end : 2012-01-22 01:02:53 


...

/home/luvil/.local/lib/python3.8/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


********** Luna **********
start : 2011-03-08 23:01:52, end : 2012-11-21 15:00:47 


/home/luvil/.local/lib/python3.8/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


********** Zaja **********
start : 2010-02-11 20:01:07, end : 2011-09-17 09:01:49 


/home/luvil/.local/lib/python3.8/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


### NEW red deer from VALAIS : unique CSV file with every animals

In [4]:
workDir = "/home/luvil/IE-OFEV/new_cerfs_valais_2025"
overwrite = True
allDat = pd.read_csv(os.path.join(workDir, "GPS_fixes_1H_bereinigt_sigb.csv"),sep=";",decimal="." )

c=1
for animalName in set(allDat["id"]):
    print(f"********** {animalName} **********")
    tmp=allDat.loc[allDat["id"]==animalName].copy()
    tmp.dropna(axis=0, subset="longitude", inplace=True)
   # tmp.dropna(axis=0, subset=["x_lv95","y_lv95"], inplace=True)
    tmp["dateTime"]=pd.to_datetime(tmp.UTC_Date+" "+tmp.UTC_Time, format="%d.%m.%Y %H:%M:%S")
    tmp["year"]=pd.Series(tmp.dateTime).dt.year
    tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
    tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
    tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
    tmp["sexe"]="unknown"
    nbYear=tmp["year"].unique().tolist()


    subset_cerf=gp.GeoDataFrame({"prenom":tmp.id,
                                "sexe":tmp.sexe,
                                "annee":tmp.year,
                                "mois":tmp.mois,
                                "saison":tmp.saison, 
                                "heure":tmp.heure,
                                "dateTime":tmp.dateTime,
                                 "geometry":gp.points_from_xy(tmp["longitude"], tmp["latitude"]),
                                "UTC_DATE":tmp.dateTime.dt.date,
                                "UTC_TIME":tmp.dateTime.dt.time
                                },
                                crs="EPSG:4326").reset_index(drop=True).sort_values(by="dateTime", axis=0)
    del(tmp)
    subset_cerf.to_crs(2056, inplace=True) 

    print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
    subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
    subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
    #print(subset_cerf["deerYear"].value_counts())


    gb = subset_cerf.groupby(['prenom', 'deerYear'])
    agregat=gb.agg(
    nbFixes=('dateTime', "count"),
    startDT=('dateTime', "min"),
    endDT=('dateTime', "max")
    ).reset_index()
    
    if c==1:
        finalStat=agregat.copy()
    else:
        finalStat=pd.concat([finalStat, agregat], ignore_index=True)
    c=c+1


    resultPath=os.path.join(workDir, str(animalName))
    try:
        os.mkdir(resultPath)
    except OSError as error:
        print(error)

    for yearTest in subset_cerf["deerYear"].unique():
        subYear=subset_cerf.loc[subset_cerf["deerYear"]==yearTest].reset_index(drop=True)
        subYear.sort_values(by="dateTime", axis=0,inplace=True)
        csvPath=os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv")
        if not os.path.isfile(csvPath) or overwrite:
            subYear.to_csv(os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv"), index=False)   
        else:
            print("file already exists : no overwriting")
    
    csvPath=os.path.join(resultPath,f"{animalName}_allFixes_formatted.csv")
    if not os.path.isfile(csvPath) or overwrite:
        subset_cerf.to_csv(os.path.join(resultPath,f"{animalName}_allFixes_formatted.csv"), index=False)   
        subset_cerf.drop(["UTC_DATE", "UTC_TIME"],axis=1).to_file(os.path.join(workDir,"RED_DEER_VALAIS_trial2_allFixes_formatted.gpkg"), driver='GPKG', layer=str(animalName))
        
    else:
        print("file already exists : no overwriting")

finalStat["Duration_days"]=(finalStat["endDT"]-finalStat["startDT"]).dt.days
finalStat["is_full_year"]=finalStat["Duration_days"].apply(lambda x : "yes" if x>=364 else "no")
finalStat.to_csv(os.path.join(workDir,"RED_DEER_VS_2_2_fixes_counts.csv"), index=False)

********** ID164 **********
start : 2019-03-22 23:00:39, end : 2019-10-02 10:00:10 
[Errno 17] File exists: '/home/luvil/IE-OFEV/new_cerfs_valais_2025/ID164'
********** ID133 **********
start : 2019-01-22 23:00:12, end : 2020-09-26 18:36:03 
[Errno 17] File exists: '/home/luvil/IE-OFEV/new_cerfs_valais_2025/ID133'
********** ID012 **********
start : 2018-01-08 23:00:41, end : 2019-12-07 22:00:46 
[Errno 17] File exists: '/home/luvil/IE-OFEV/new_cerfs_valais_2025/ID012'
********** ID158 **********
start : 2020-01-10 23:00:58, end : 2020-10-23 20:00:17 
[Errno 17] File exists: '/home/luvil/IE-OFEV/new_cerfs_valais_2025/ID158'
********** ID003 **********
start : 2017-12-31 23:00:29, end : 2019-11-21 22:00:10 
[Errno 17] File exists: '/home/luvil/IE-OFEV/new_cerfs_valais_2025/ID003'
********** ID068 **********
start : 2018-04-11 23:00:13, end : 2020-03-07 22:00:08 
[Errno 17] File exists: '/home/luvil/IE-OFEV/new_cerfs_valais_2025/ID068'
********** ID102 **********
start : 2019-01-18 23:01

In [ ]:
### NOTE : for yano, I denoised obvious false points #####
denoiseFunctions.detectHighDist(subset_cerf, maxDist=4000)
## the function returned :[0, 5265, 5266, 5269, 5280, 5282, 5284]
## therefore, I removed all the points from 5265 to the end (since subset_cerf.shape[0] returned 5284 rows)
tmp=subset_cerf.iloc[:5264]
subset_cerf=tmp.copy()
subset_cerf.to_csv(os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv"), index=False)   

### "Nouveau" red deer from Bern :: single file per animal

In [ ]:
animalTest = "Hope"
fileDataName="HOPE_GPS_Collar37181_20230629100702.csv"
filePath=os.path.join(workDir, "new_cerfs_Bern",fileDataName)
print(filePath)
tmp=pd.read_csv(filePath, sep=";", encoding="iso8859_2", lineterminator="\r",decimal=",",skip_blank_lines=True)
tmp.dropna(axis=0, subset="UTC_Date", inplace=True)
tmp.dropna(axis=0, subset=["ECEF_X [m]","ECEF_Y [m]"], inplace=True)
tmp["prenom"]=animalTest
tmp["dateTime"]=pd.to_datetime(tmp.UTC_Date+" "+tmp.UTC_Time, format="%d.%m.%Y %H:%M:%S")
tmp["year"]=pd.Series(tmp.dateTime).dt.year
tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
tmp["sexe"]=tmp["prenom"].apply(mapSexASB)
nbYear=tmp["year"].unique().tolist()

subset_cerf=gp.GeoDataFrame({"prenom":tmp.prenom,
                             "sexe":tmp.sexe,
                             "annee":tmp.year,
                             "mois":tmp.mois,
                            "saison":tmp.saison, 
                            "heure":tmp.heure,
                            "dateTime":tmp.dateTime,
                            "geometry":gp.points_from_xy(tmp["Longitude [°]"], tmp["Latitude [°]"]),
                            "UTC_DATE":tmp.dateTime.dt.date,
                            "UTC_TIME":tmp.dateTime.dt.time
                            },
                            crs="EPSG:4326").reset_index(drop=True)
subset_cerf.to_crs(epsg=2056, inplace=True)
del(tmp)
print(f"********** {animalTest} **********")
print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
print(subset_cerf["deerYear"].value_counts())
resultPath=os.path.join(workDir, animalTest)
try:
    os.mkdir(resultPath)
except OSError as error:
    print(error)
csvPath=os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv")
if not os.path.isfile(csvPath):
    subset_cerf.to_csv(os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv"), index=False)   
else:
    print("file already exists")

/home/loreto/Documents/FractalDeer/Nouveau/ZIMA_GPS_Collar37172_20230629100252.csv


/tmp/ipykernel_5597/879820633.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  tmp=pd.read_csv(filePath, sep=";", encoding="iso8859_2", lineterminator="\r",decimal=",",skip_blank_lines=True)


********** Zima **********
start : 2022-02-09 18:32:02, end : 2023-06-29 04:00:10 
deerYear_2022-2023    17462
deerYear_2023-2024     5703
deerYear_2021-2022      920
Name: deerYear, dtype: int64
[Errno 17] File exists: '/home/loreto/Documents/FractalDeer/Zima'
file already exists


### red deer from ZH

In [ ]:
animalTest="Erika"

fileDataName="ID04_Dario_Male_GPS_Collar21103_20230601.csv"
filePath=os.path.join(workDir, "GPSdataZH","Positionen_3TageNachBesenderung_bis_20230531_WILMA-ZHAW",fileDataName)
tmp=pd.read_csv(filePath, sep=";", encoding="iso8859_2", lineterminator="\r",decimal=",",skip_blank_lines=True,skiprows=[1])
tmp.dropna(axis=0, subset="UTC_Date", inplace=True)
tmp.dropna(axis=0, subset=["ECEF_X [m]","ECEF_Y [m]"], inplace=True)
tmp["prenom"]=animalTest
tmp["dateTime"]=pd.to_datetime(tmp.UTC_Date+" "+tmp.UTC_Time, format="%d.%m.%Y %H:%M:%S")
tmp["year"]=pd.Series(tmp.dateTime).dt.year
tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
tmp["sexe"]=tmp["prenom"].apply(mapSexZH)
nbYear=tmp["year"].unique().tolist()



subset_cerf=gp.GeoDataFrame({"prenom":tmp.prenom,
                             "sexe":tmp.sexe,
                             "annee":tmp.year,
                             "mois":tmp.mois,
                            "saison":tmp.saison, 
                            "heure":tmp.heure,
                            "dateTime":tmp.dateTime,
                            "geometry":gp.points_from_xy(tmp["Longitude [°]"], tmp["Latitude [°]"]),
                            "UTC_DATE":tmp.dateTime.dt.date,
                            "UTC_TIME":tmp.dateTime.dt.time
                            },
                            crs="EPSG:4326").reset_index(drop=True)
subset_cerf.to_crs(epsg=2056, inplace=True)
subset_cerf["x_coord"]=subset_cerf.geometry.x
subset_cerf["y_coord"]=subset_cerf.geometry.y
print(f"********** {animalTest} **********")
print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
del(tmp)
print(subset_cerf["deerYear"].value_counts())
resultPath=os.path.join(workDir, animalTest)
try:
    os.mkdir(resultPath)
except OSError as error:
    print(error)
csvPath=os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv")
if not os.path.isfile(csvPath):
    subset_cerf.to_csv(os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv"), index=False)   
else:
    print("file already exists")


********** Dario **********
start : 2022-03-31 00:00:36, end : 2023-05-25 15:00:35 
deerYear_2022-2023    7952
deerYear_2023-2024    2043
Name: deerYear, dtype: int64
[Errno 17] File exists: '/home/lucas/FractalDeer_project/FractalDeer/Dario'
file already exists


### red deer from versoix : a unique file with every animals

In [ ]:

animalTest = "Neige"


tmp=pd.read_csv("all_cerfs_Versoix_valides_2021.csv")
tmp["dateTime"]=pd.to_datetime(tmp.UTC_DATE+" "+tmp.UTC_TIME, format="%Y/%m/%d %H:%M:%S")
tmp["year"]=pd.Series(tmp.dateTime).dt.year
tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
tmp["sexe"]=tmp["prenom"].apply(mapSexVersoix)
nbYear=tmp["annee"].unique().tolist()

subset_cerf_1=tmp.loc[tmp["prenom"]==animalTest].reset_index()

subset_cerf=gp.GeoDataFrame({"prenom":subset_cerf_1.prenom,
                            "sexe":subset_cerf_1.sexe,
                             "annee":subset_cerf_1.year,
                             "mois":subset_cerf_1.mois,
                            "saison":subset_cerf_1.saison, 
                            "heure":subset_cerf_1.heure,
                            "dateTime":subset_cerf_1.dateTime,
                            "geometry":gp.points_from_xy(subset_cerf_1["LONGITUDE"], subset_cerf_1["LATITUDE"]),
                            "UTC_DATE":subset_cerf_1.dateTime.dt.date,
                            "UTC_TIME":subset_cerf_1.dateTime.dt.time,
                            },
                            crs="EPSG:4326").reset_index()
subset_cerf.to_crs(2056, inplace=True) 
subset_cerf["x_coord"]=subset_cerf.geometry.x
subset_cerf["y_coord"]=subset_cerf.geometry.y
print(f"********** {animalTest} **********")
print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
del(subset_cerf_1)
del(tmp)

print(subset_cerf["deerYear"].value_counts())

resultPath=os.path.join(workDir, animalTest)
try:
    os.mkdir(resultPath)
except OSError as error:
    print(error)
csvPath=os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv")
if not os.path.isfile(csvPath):
    subset_cerf.to_csv(os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv"), index=False)   
else:
    print("file already exists")


********** Neige **********
start : 2012-12-13 00:01:04, end : 2016-05-14 01:02:18 
deerYear_2015-2016    4722
deerYear_2014-2015    3193
deerYear_2013-2014    2826
deerYear_2016-2017    1010
deerYear_2012-2013     635
Name: deerYear, dtype: int64


### red deer from haute-savoie

In [ ]:
animalTest="45568"
try:
    tmp=pd.read_csv(os.path.join("/home/lucas/cerfArves_3",animalTest,f"dataset_cerf_{animalTest}_denoised.csv"))
except FileNotFoundError:
    print("file not found !")

subset_cerf=gp.GeoDataFrame({"prenom":tmp.AnimalID.apply(lambda x : str(x)),
                            "sexe":tmp.sexe,
                            "annee":tmp.annee,
                             "mois":tmp.mois,
                            "saison":tmp.saison, 
                            "dateTime":pd.to_datetime(tmp.dateTime, format="%Y-%m-%d %H:%M:%S"),
                            "geometry":gp.points_from_xy(tmp["coord_X_L93"], tmp["coord_Y_L93"]),
                            "deerYear":tmp.annee_cerf,
                            },
                            crs="EPSG:2154").reset_index()

subset_cerf["UTC_DATE"]=subset_cerf.dateTime.dt.date
subset_cerf["UTC_TIME"]=subset_cerf.dateTime.dt.time
subset_cerf["x_coord"]=subset_cerf.geometry.x
subset_cerf["y_coord"]=subset_cerf.geometry.y
subset_cerf["deerYear"]=tmp["annee_cerf"]
subset_cerf["jourNuit"]=tmp["periodeJour"]

resultPath=os.path.join(workDir, animalTest)
try:
    os.mkdir(resultPath)
except OSError as error:
    print(error)
csvPath=os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv")
if not os.path.isfile(csvPath):
    subset_cerf.to_csv(os.path.join(resultPath,f"{animalTest}_allFixes_formatted.csv"), index=False)   
else:
    print("file already exists")

### if you want to save the fixes for a given deer-year in a separate file

In [2]:
animalTest="Bajo"
#yearTest=["deerYear_2021-2022","deerYear_2022-2023"]

yearTest=["deerYear_2016-2017"]
try:
    tmp=pd.read_csv(os.path.join(workDir, animalTest, f"{animalTest}_allFixes_formatted.csv"))
    subset_cerf=gp.GeoDataFrame(tmp, geometry=gp.GeoSeries.from_wkt(tmp.geometry),crs=2056)
    subset_cerf=subset_cerf.loc[subset_cerf["deerYear"].isin(yearTest)].reset_index(drop=True)
    subset_cerf.sort_values(by="dateTime", axis=0,inplace=True)
except FileNotFoundError as fnf:
    print("this file does not exist.. did you forget to generate it ?")

print(subset_cerf["prenom"].value_counts())
resultPath=os.path.join(workDir, "selected_residents")
try:
    os.mkdir(resultPath)
except OSError as error:
    print(error)
csvPath=os.path.join(resultPath,f"{animalTest}_{yearTest}_allFixes_formatted.csv")
if not os.path.isfile(csvPath):
    subset_cerf.to_csv(os.path.join(resultPath,f"{animalTest}_{'_'.join(yearTest)}_allFixes_formatted.csv"), index=False)   
else:
    print("file already exists")

Bajo    13596
Name: prenom, dtype: int64
[Errno 17] File exists: '/home/lucas/FractalDeer_project/FractalDeer/selected_residents'


In [ ]:
import geopandas as gpd
#workDir = "smb://fs.hes.adhes.hesge.ch/srv/RDPS/HEPIA/GN-RDPS/UDRN/01_PROJETS/Fischer/2_En%20cours/Avec%20SAGEX/IE-OFEV_134'226/cerf_versoix"
workDir = "./"
c=1
overwrite=True
for root, dirs, files in os.walk(workDir):
    for file in files:
        if file.lower().endswith("jura.csv"):
            tmp = gpd.read_file(file)
            tmp["dateTime"]=pd.to_datetime(tmp.UTC_DATE+" "+tmp.UTC_TIME, format="%Y/%m/%d %H:%M:%S")

            gb = subset_cerf.groupby(['prenom', 'deerYear'])
            agregat=gb.agg(
            nbFixes=('dateTime', "count"),
            startDT=('dateTime', "min"),
            endDT=('dateTime', "max")
            ).reset_index()
            
            if c==1:
                finalStat=agregat.copy()
            else:
                finalStat=pd.concat([finalStat, agregat], ignore_index=True)
            c=c+1

            animalName = tmp["prenom"].unique()[0]
            resultPath=os.path.join(root, str(animalName))
            try:
                os.mkdir(resultPath)
            except OSError as error:
                print(error)

            for yearTest in subset_cerf["deerYear"].unique():
                subYear=subset_cerf.loc[subset_cerf["deerYear"]==yearTest].reset_index(drop=True)
                subYear.sort_values(by="dateTime", axis=0,inplace=True)
                csvPath=os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv")
                if not os.path.isfile(csvPath) or overwrite:
                    subYear.to_csv(os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv"), index=False)   
                else:
                    print("file already exists : no overwriting")
            

finalStat["Duration_days"]=(finalStat["endDT"]-finalStat["startDT"]).dt.days
finalStat["is_full_year"]=finalStat["Duration_days"].apply(lambda x : "yes" if x>=364 else "no")
finalStat.to_csv(os.path.join(workDir,"RED_DEER_GE_fixes_counts.csv"), index=False)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
file=pd.read_csv('/home/luvil/IE-OFEV/jura_selected/Bimodal_pattern/lila deerYear 2012-2013 stepSize 754 autocorrelation timeSeriesKmeans 2 classes REFINED RATIO_ENDNSD.csv')

shapefile=gpd.GeoDataFrame(file, geometry=gpd.points_from_xy(file["x"], file["y"]), crs="epsg:2056").reset_index()
jura=gpd.read_file('/home/luvil/IE-OFEV/bioregions_11mars25/Jura_only.shp').reset_index()
inJura=[]
for index, row in shapefile.iterrows():
    #inJura.append(jura.contains(row['geometry']))   
    inJura.append(any(jura.contains(row.geometry)))
shapefile["jura"]=inJura

outFile=shapefile.to_csv('/home/luvil/IE-OFEV/jura_selected/Bimodal_pattern/lila deerYear 2012-2013 stepSize 754 autocorrelation timeSeriesKmeans 2 classes REFINED RATIO_ENDNSD_JURA.csv')

### new red deer from versoix (october 2025)

In [25]:
animalName="Bibi"
animalSex="male"
filePath="/home/luvil/IE-OFEV/cerf_versoix_2/Bibi/GPS_Collar95529_20251002143752.csv"
resultPath="/home/luvil/IE-OFEV/cerf_versoix_2_parsed/"
overwrite=True
tmp=pd.read_csv(filePath)
tmp.dropna(axis=0, subset="UTC_Date", inplace=True)
tmp.dropna(axis=0, subset=["ECEF_X [m]","ECEF_Y [m]"], inplace=True)
tmp["prenom"]=animalName
tmp["dateTime"]=pd.to_datetime(tmp.UTC_Date+" "+tmp.UTC_Time, format="%d.%m.%Y %H:%M:%S")
tmp["year"]=pd.Series(tmp.dateTime).dt.year
tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
tmp["sexe"]=animalSex
nbYear=tmp["year"].unique().tolist()



subset_cerf=gp.GeoDataFrame({"prenom":tmp.prenom,
                             "sexe":tmp.sexe,
                             "annee":tmp.year,
                             "mois":tmp.mois,
                            "saison":tmp.saison, 
                            "heure":tmp.heure,
                            "dateTime":tmp.dateTime,
                            "geometry":gp.points_from_xy(tmp["Longitude [°]"], tmp["Latitude [°]"]),
                            "UTC_DATE":tmp.dateTime.dt.date,
                            "UTC_TIME":tmp.dateTime.dt.time
                            },
                            crs="EPSG:4326").reset_index(drop=True)
subset_cerf.to_crs(epsg=2056, inplace=True)
subset_cerf["x_coord"]=subset_cerf.geometry.x
subset_cerf["y_coord"]=subset_cerf.geometry.y
print(f"********** {animalName} **********")
print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
del(tmp)



gb = subset_cerf.groupby(['prenom', 'deerYear'])
finalStat=gb.agg(
nbFixes=('dateTime', "count"),
startDT=('dateTime', "min"),
endDT=('dateTime', "max")
).reset_index()

resultPath=os.path.join(resultPath, animalName)
try:
    os.mkdir(resultPath)
except OSError as error:
    print(error)

for yearTest in subset_cerf["deerYear"].unique():
    subYear=subset_cerf.loc[subset_cerf["deerYear"]==yearTest].reset_index(drop=True)
    subYear.sort_values(by="dateTime", axis=0,inplace=True)
    csvPath=os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv")
    if not os.path.isfile(csvPath) or overwrite:
        subYear.to_csv(os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv"), index=False)   
    else:
        print("file already exists : no overwriting")

csvPath=os.path.join(resultPath,f"{animalName}_allFixes_formatted.csv")
if not os.path.isfile(csvPath) or overwrite:
    subset_cerf.to_csv(csvPath, index=False)   
    
else:
    print("file already exists : no overwriting")

finalStat["Duration_days"]=(finalStat["endDT"]-finalStat["startDT"]).dt.days
finalStat["is_full_year"]=finalStat["Duration_days"].apply(lambda x : "yes" if x>=364 else "no")
finalStat.to_csv(os.path.join(resultPath,f"{animalName}_fixes_counts.csv"), index=False)
print("***** PARSING DONE ******")

********** Bibi **********
start : 2024-11-03 23:01:23, end : 2025-09-29 03:00:11 
***** PARSING DONE ******


In [ ]:

import pandas as pd
import os
from io import StringIO
import geopandas as gp
import pytz
import astral
from astral.sun import sun
import locale
from datetime import datetime, date,time
import sys
import numpy as np


def mapSeasons(monthValue):
    corr={1:"Decembre-Fevrier",
          2:"Decembre-Fevrier",
          3:"Mars-Mai",
          4:"Mars-Mai",
          5:"Mars-Mai",
          6:"Juin-Aout",
          7:"Juin-Aout",
          8:"Juin-Aout",
          9:"Septembre-Novembre",
          10:"Septembre-Novembre",
          11:"Septembre-Novembre",
          12:"Decembre-Fevrier"}
    return corr[monthValue]   

def getDeerYear(gdf_cerf,yearList:list):
  deerYearVector=pd.Series(["NA" for i in range(len(gdf_cerf))])
  yearList.append(yearList[-1]+1)
  for i in yearList:
    if pd.to_datetime(i, format="%Y").is_leap_year:
        deerYear=[pd.Timestamp(i-1,3,1),pd.Timestamp(i,2,29)+pd.Timedelta(days=1)]
    else:
        deerYear=[pd.Timestamp(i-1,3,1),pd.Timestamp(i,2,28)+pd.Timedelta(days=1)]

    #deerYear=[pd.Timestamp(i-1,6,1),pd.Timestamp(i,5,31)+pd.Timedelta(days=1)]    
    
    for index, row in gdf_cerf.iterrows():
         if deerYear[0]<row.dateTime<=deerYear[1]:
              deerYearVector.iloc[index]=f"deerYear_{i-1}-{i}"
  return deerYearVector

def getSunPeriod(gdf_cerf:gp.GeoDataFrame, country:str):
    periodList=[]
    longLat=gdf_cerf.geometry.to_crs("epsg:4326")
    for index, row in gdf_cerf.iterrows():

        if country=="Switzerland":
            local_tz = pytz.timezone('Europe/Zurich')
            loc_cerf=astral.LocationInfo("Suisse",region="Switzerland",timezone="Europe/Zurich",latitude=longLat.iloc[index].y, longitude=longLat.iloc[index].x)
        elif country=="France":
            local_tz = pytz.timezone('Europe/Paris')
            loc_cerf=astral.LocationInfo("Haute-Savoie",region="France",timezone="Europe/Paris",latitude=longLat.iloc[index].y, longitude=longLat.iloc[index].x)
        else:
            print("no country was specified")
            raise Exception


        s = sun(loc_cerf.observer, date=row.dateTime)
        if local_tz.localize(row.dateTime)<=s["dawn"]:
            period="nuit"
        elif local_tz.localize(row.dateTime)<=s["sunrise"]:
            period="nuit"
        elif local_tz.localize(row.dateTime)<=s["sunset"]:
            period="jour"
        elif local_tz.localize(row.dateTime)<=s["dusk"]:
            period="jour"
        else:
            period="nuit"
        periodList.append(period)
    return periodList

### environment parameters ###
workDir="/home/luvil/IE-OFEV/cerf_grisons_3/"

overwrite=True

###############################
mapSex={"f":"femelle", "m":"male"}


allDat=pd.read_csv(os.path.join(workDir, "gps_rae.csv"))

c=1
for animalName in set(allDat["ID"]):
    print(f"********** {animalName} **********")
    tmp=allDat.loc[allDat["ID"]==animalName].copy()
   # tmp.dropna(axis=0, subset="acquisition_time", inplace=True)
    tmp.dropna(axis=0, subset=["x_lv95","y_lv95"], inplace=True)
    tmp["prenom"]=animalName
    tmp["dateTime"]=pd.to_datetime(tmp.acquisition_time, format="%Y-%m-%d %H:%M:%S")
    tmp["year"]=pd.Series(tmp.dateTime).dt.year
    tmp["mois"]=pd.Series(tmp.dateTime).dt.month_name(locale=locale.getlocale())
    tmp["heure"]=pd.Series(tmp.dateTime).dt.hour
    tmp["saison"]=tmp["dateTime"].dt.month.apply(mapSeasons)
    tmp["sexe"]=tmp["Sex"].replace(mapSex)
    nbYear=tmp["year"].unique().tolist()


    subset_cerf=gp.GeoDataFrame({"prenom":tmp.prenom,
                                "sexe":tmp.sexe,
                                "annee":tmp.year,
                                "mois":tmp.mois,
                                "saison":tmp.saison, 
                                "heure":tmp.heure,
                                "dateTime":tmp.dateTime,
                                 "geometry":gp.points_from_xy(tmp["longitude"], tmp["latitude"]),
                                "UTC_DATE":tmp.dateTime.dt.date,
                                "UTC_TIME":tmp.dateTime.dt.time
                                },
                                crs="EPSG:4326").reset_index(drop=True).sort_values(by="dateTime", axis=0)
    del(tmp)
    subset_cerf.to_crs(epsg=2056, inplace=True)
    print(f"start : {subset_cerf.iloc[0].dateTime}, end : {subset_cerf.iloc[-1].dateTime} ")
    subset_cerf["deerYear"]=getDeerYear(subset_cerf, yearList=nbYear)
    subset_cerf["jourNuit"]=getSunPeriod(subset_cerf, country="Switzerland")
    #print(subset_cerf["deerYear"].value_counts())


    gb = subset_cerf.groupby(['prenom', 'deerYear'])
    agregat=gb.agg(
    nbFixes=('dateTime', "count"),
    startDT=('dateTime', "min"),
    endDT=('dateTime', "max")
    ).reset_index()
    
    if c==1:
        finalStat=agregat.copy()
    else:
        finalStat=pd.concat([finalStat, agregat], ignore_index=True)
    c=c+1


    resultPath=os.path.join(workDir, str(animalName))
    try:
        os.mkdir(resultPath)
    except OSError as error:
        print(error)

    for yearTest in subset_cerf["deerYear"].unique():
        subYear=subset_cerf.loc[subset_cerf["deerYear"]==yearTest].reset_index(drop=True)
        subYear.sort_values(by="dateTime", axis=0,inplace=True)
        csvPath=os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv")
        if not os.path.isfile(csvPath) or overwrite:
            subYear.to_csv(os.path.join(resultPath,f"{str(animalName)}_{yearTest}.csv"), index=False)   
        else:
            print("file already exists : no overwriting")
    
    csvPath=os.path.join(resultPath,f"{animalName}_allFixes_formatted.csv")
    if not os.path.isfile(csvPath) or overwrite:
        subset_cerf.to_csv(csvPath, index=False)   
        subset_cerf.drop(["UTC_DATE", "UTC_TIME"],axis=1).to_file(os.path.join(workDir,"RED_DEER_GRISONS_2_allFixes_formatted.gpkg"), driver='GPKG', layer=str(animalName))
        
    else:
        print("file already exists : no overwriting")

finalStat["Duration_days"]=(finalStat["endDT"]-finalStat["startDT"]).dt.days
finalStat["is_full_year"]=finalStat["Duration_days"].apply(lambda x : "yes" if x>=364 else "no")
finalStat.to_csv(os.path.join(workDir,"RED_DEER_GRISONS_2_fixes_counts.csv"), index=False)
print("***** PARSING DONE ***** ")

********** 21001 **********
start : 2014-03-13 00:01:42, end : 2015-12-15 21:01:02 
[Errno 17] File exists: '/home/luvil/IE-OFEV/cerf_grisons_2/21001'
********** 21002 **********
start : 2014-03-13 00:01:42, end : 2014-10-23 21:02:54 
[Errno 17] File exists: '/home/luvil/IE-OFEV/cerf_grisons_2/21002'
********** 21003 **********
start : 2014-03-14 00:01:44, end : 2014-06-25 21:00:59 
[Errno 17] File exists: '/home/luvil/IE-OFEV/cerf_grisons_2/21003'
********** 21004 **********
start : 2014-03-12 00:01:54, end : 2015-07-24 18:01:29 
[Errno 17] File exists: '/home/luvil/IE-OFEV/cerf_grisons_2/21004'
********** 21005 **********
start : 2014-03-13 00:01:43, end : 2016-03-23 21:00:43 
[Errno 17] File exists: '/home/luvil/IE-OFEV/cerf_grisons_2/21005'
********** 21006 **********
start : 2014-03-15 00:01:18, end : 2014-08-23 21:02:11 
[Errno 17] File exists: '/home/luvil/IE-OFEV/cerf_grisons_2/21006'
********** 21008 **********
start : 2014-03-22 00:02:25, end : 2016-01-11 21:02:42 
[Errno 17]